In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from scipy.special import ellipk

from ipywidgets import (
    FloatSlider, HTML, HTMLMath,
    VBox, HBox, Layout
)

from IPython.display import display

# ============================================================
# JACOBI NOME AND COMPLEMENTARY NOME
#
# q(k)  = exp(-pi K'/K)
# qc(k) = exp(-pi K/K')
#
# scipy.special.ellipk uses m = k^2
# ============================================================

plt.ioff()

# ============================================================
# JUPYTER / BINDER DISPLAY SETTINGS
# ============================================================

display(HTML("""
<style>

.container {
    width:98% !important;
    max-width:none !important;
}

.output_area,
.output_subarea {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.output_scroll {
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
    box-shadow:none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width:none !important;
    height:auto !important;
    max-height:none !important;
    overflow:visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow:visible !important;
    resize:none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display:none !important;
}

.nome-title {
    font-family:Arial, sans-serif;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

.nome-label {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
}

.nome-value {
    font-family:Arial, sans-serif;
    font-size:14px;
    font-weight:bold;
    color:#0b3d91;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1200px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="nome-title" style="margin-bottom:8px;">
Jacobi Nome and Complementary Nome
</div>

<div style="margin-bottom:5px;">
The Jacobi nome q(k) and its complementary function q<sub>c</sub>(k)
are defined in terms of the complete elliptic integrals K and K′.
</div>

<div style="margin-bottom:5px;">
The notebook compares the exact expressions with the truncated series
approximations presented in the theory. The q-series is used for
0 ≤ k ≤ 1/√2, while the complementary series is used for
1/√2 ≤ k ≤ 1.
</div>

<div>
<b>This notebook:</b> shows q(k), q<sub>c</sub>(k), the ratios K′/K
and K/K′, and verifies numerically the identity
ln(q) ln(q<sub>c</sub>) = π².
</div>

</div>
""")

# ============================================================
# DEFINITIONS
# ============================================================

q_definition = HTMLMath(
    value=(
        r'\('
        r'q(k)'
        r'='
        r'\exp\left(-\pi\dfrac{K^{\prime}}{K}\right)'
        r'\)'
    )
)

qc_definition = HTMLMath(
    value=(
        r'\('
        r'q_c(k)'
        r'='
        r'\exp\left(-\pi\dfrac{K}{K^{\prime}}\right)'
        r'\)'
    )
)

identity_definition = HTMLMath(
    value=(
        r'\('
        r'\ln(q)\ln(q_c)=\pi^2'
        r'\)'
    )
)

definitions_row = HBox(
    [
        q_definition,
        qc_definition,
        identity_definition
    ],
    layout=Layout(
        width='1150px',
        gap='30px',
        align_items='center',
        overflow='visible'
    )
)

# ============================================================
# SLIDER
# ============================================================

slider_style = {
    'description_width': '0px'
}

k_slider = FloatSlider(
    min=0.01,
    max=0.99,
    step=0.01,
    value=0.50,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=Layout(width='260px')
)

k_label = HTML(
    '<div class="nome-label">Elliptic modulus k:</div>',
    layout=Layout(
        width='150px',
        min_width='150px'
    )
)

k_value = HTML(
    '<div class="nome-value">0.50</div>',
    layout=Layout(
        width='65px',
        min_width='65px',
        margin='0px 0px 0px 6px'
    )
)

k_row = HBox(
    [k_label, k_slider, k_value],
    layout=Layout(
        width='500px',
        height='42px',
        align_items='center'
    )
)

# ============================================================
# PARAMETER PANEL
# ============================================================

parameter_panel = VBox(
    [
        HTML("""
        <div class="nome-title" style="margin-bottom:8px;">
            Parameter
        </div>
        """),
        k_row
    ],
    layout=Layout(
        width='520px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# CURRENT VALUES
# ============================================================

kp_math = HTMLMath()
K_math = HTMLMath()
Kp_math = HTMLMath()

q_math = HTMLMath()
qc_math = HTMLMath()

ratio1_math = HTMLMath()
ratio2_math = HTMLMath()

identity_math = HTMLMath()

values_panel = VBox(
    [
        HTML("""
        <div class="nome-title" style="margin-bottom:8px;">
            Current Values
        </div>
        """),

        HBox(
            [kp_math, K_math, Kp_math],
            layout=Layout(
                width='600px',
                gap='18px',
                overflow='visible'
            )
        ),

        HBox(
            [q_math, qc_math],
            layout=Layout(
                width='580px',
                gap='25px',
                overflow='visible'
            )
        ),

        HBox(
            [ratio1_math, ratio2_math],
            layout=Layout(
                width='580px',
                gap='25px',
                overflow='visible'
            )
        ),

        identity_math
    ],
    layout=Layout(
        width='640px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# TOP ROW
# ============================================================

top_row = HBox(
    [parameter_panel, values_panel],
    layout=Layout(
        width='1180px',
        gap='15px',
        align_items='stretch',
        overflow='visible'
    )
)

# ============================================================
# FUNCTIONS
# ============================================================

def elliptic_values(k):
    m = k**2
    kp = np.sqrt(1.0 - k**2)

    K = ellipk(m)
    Kp = ellipk(1.0 - m)

    q = np.exp(-np.pi * Kp / K)
    qc = np.exp(-np.pi * K / Kp)

    return kp, K, Kp, q, qc


def epsilon_q(k):
    fourth_root = (1.0 - k**2)**0.25

    return 0.5 * (
        (1.0 - fourth_root)
        /
        (1.0 + fourth_root)
    )


def epsilon_qc(k):
    root_k = np.sqrt(k)

    return 0.5 * (
        (1.0 - root_k)
        /
        (1.0 + root_k)
    )


def q_series(k):
    e = epsilon_q(k)

    return (
        e
        + 2.0 * e**5
        + 15.0 * e**9
        + 150.0 * e**13
        + 1707.0 * e**17
    )


def qc_series(k):
    e = epsilon_qc(k)

    return (
        e
        + 2.0 * e**5
        + 15.0 * e**9
        + 150.0 * e**13
        + 1707.0 * e**17
    )

# ============================================================
# k GRID
# ============================================================

k_axis = np.linspace(
    0.01,
    0.99,
    1600
)

threshold = (
    1.0 / np.sqrt(2.0)
)

m_axis = (
    k_axis**2
)

K_axis = ellipk(
    m_axis
)

Kp_axis = ellipk(
    1.0 - m_axis
)

q_axis = np.exp(
    -np.pi * Kp_axis / K_axis
)

qc_axis = np.exp(
    -np.pi * K_axis / Kp_axis
)

ratio_Kp_K = (
    Kp_axis / K_axis
)

ratio_K_Kp = (
    K_axis / Kp_axis
)

# ============================================================
# SERIES VALUES
#
# Outside their convergence regions NaN is used so the
# corresponding curve is not drawn.
# ============================================================

q_series_axis = np.full_like(
    k_axis,
    np.nan
)

qc_series_axis = np.full_like(
    k_axis,
    np.nan
)

mask_q = (
    k_axis <= threshold
)

mask_qc = (
    k_axis >= threshold
)

q_series_axis[mask_q] = q_series(
    k_axis[mask_q]
)

qc_series_axis[mask_qc] = qc_series(
    k_axis[mask_qc]
)

# ============================================================
# FIGURE 1
# NOME FUNCTIONS
# ============================================================

fig_nome, ax_nome = plt.subplots(
    figsize=(6.0, 4.7)
)

fig_nome.canvas.header_visible = False
fig_nome.canvas.footer_visible = False
fig_nome.canvas.toolbar_visible = False

fig_nome.canvas.layout = Layout(
    width='600px',
    height='470px',
    overflow='visible'
)

ax_nome.set_title(
    'Jacobi Nome Functions',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax_nome.set_xlabel(
    'Elliptic modulus k',
    fontsize=10
)

ax_nome.set_ylabel(
    'Nome value',
    fontsize=10
)

ax_nome.set_xlim(
    0.0,
    1.0
)

ax_nome.set_ylim(
    -0.02,
    1.02
)

ax_nome.grid(
    True,
    linestyle=':',
    alpha=0.40
)

line_q, = ax_nome.plot(
    k_axis,
    q_axis,
    linewidth=2.0,
    label='q(k) exact'
)

line_qc, = ax_nome.plot(
    k_axis,
    qc_axis,
    linewidth=2.0,
    label='q_c(k) exact'
)

line_q_series, = ax_nome.plot(
    k_axis,
    q_series_axis,
    linestyle='--',
    linewidth=1.7,
    label='q(k) series'
)

line_qc_series, = ax_nome.plot(
    k_axis,
    qc_series_axis,
    linestyle='--',
    linewidth=1.7,
    label='q_c(k) series'
)

ax_nome.axvline(
    threshold,
    linestyle=':',
    linewidth=1.3
)

current_nome_line = ax_nome.axvline(
    k_slider.value,
    linestyle='--',
    linewidth=1.1
)

point_q, = ax_nome.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

point_qc, = ax_nome.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

ax_nome.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.16),
    ncol=4,
    fontsize=8,
    frameon=True
)

fig_nome.subplots_adjust(
    left=0.12,
    right=0.97,
    top=0.90,
    bottom=0.25
)

# ============================================================
# FIGURE 2
# K'/K AND K/K'
# ============================================================

fig_ratio, ax_ratio = plt.subplots(
    figsize=(6.0, 4.7)
)

fig_ratio.canvas.header_visible = False
fig_ratio.canvas.footer_visible = False
fig_ratio.canvas.toolbar_visible = False

fig_ratio.canvas.layout = Layout(
    width='600px',
    height='470px',
    overflow='visible'
)

ax_ratio.set_title(
    'Ratios of Complete Elliptic Integrals',
    fontsize=14,
    fontweight='bold',
    color='#0b3d91'
)

ax_ratio.set_xlabel(
    'Elliptic modulus k',
    fontsize=10
)

ax_ratio.set_ylabel(
    'Ratio',
    fontsize=10
)

ax_ratio.set_xlim(
    0.0,
    1.0
)

ax_ratio.set_ylim(
    0.0,
    4.0
)

ax_ratio.grid(
    True,
    linestyle=':',
    alpha=0.40
)

line_ratio1, = ax_ratio.plot(
    k_axis,
    ratio_Kp_K,
    linewidth=2.0,
    label='K′ / K'
)

line_ratio2, = ax_ratio.plot(
    k_axis,
    ratio_K_Kp,
    linewidth=2.0,
    label='K / K′'
)

ax_ratio.axvline(
    threshold,
    linestyle=':',
    linewidth=1.3
)

current_ratio_line = ax_ratio.axvline(
    k_slider.value,
    linestyle='--',
    linewidth=1.1
)

point_ratio1, = ax_ratio.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

point_ratio2, = ax_ratio.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

ax_ratio.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.16),
    ncol=2,
    fontsize=9,
    frameon=True
)

fig_ratio.subplots_adjust(
    left=0.12,
    right=0.97,
    top=0.90,
    bottom=0.25
)

# ============================================================
# FIGURES ROW
# ============================================================

figures_row = HBox(
    [
        fig_nome.canvas,
        fig_ratio.canvas
    ],
    layout=Layout(
        width='1220px',
        gap='15px',
        align_items='flex-start',
        overflow='visible'
    )
)

# ============================================================
# SERIES CHECK PANEL
# ============================================================

series_exact_math = HTMLMath()
series_approx_math = HTMLMath()
series_error_math = HTMLMath()

series_panel = VBox(
    [
        HTML("""
        <div class="nome-title" style="margin-bottom:8px;">
            Series Approximation
        </div>
        """),

        series_exact_math,
        series_approx_math,
        series_error_math
    ],
    layout=Layout(
        width='570px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# INTERPRETATION PANEL
# ============================================================

interpretation_panel = HTML("""
<div style="
    width:670px;
    padding:10px 14px;
    border:1px solid #d7c7e5;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.52;
    box-sizing:border-box;
">

<div style="
    color:#6f3fa0;
    font-size:17px;
    font-weight:bold;
    margin-bottom:6px;
">
Interpretation
</div>

<div style="margin-bottom:6px;">
The value k = 1/√2 separates the two convergence regions used
for the truncated nome series.
</div>

<div style="margin-bottom:6px;">
For small k, q(k) is small while q<sub>c</sub>(k) is large.
As k approaches 1 the situation is reversed.
</div>

<div>
The ratios K′/K and K/K′ determine the exponential behavior of
the nome functions and play an important role in calculations
associated with elliptic functions and elliptic-filter design.
</div>

</div>
""")

# ============================================================
# BOTTOM ROW
# ============================================================

bottom_row = HBox(
    [
        series_panel,
        interpretation_panel
    ],
    layout=Layout(
        width='1160px',
        gap='15px',
        align_items='stretch',
        overflow='visible'
    )
)

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_notebook(change=None):

    k = k_slider.value

    kp, K, Kp, q, qc = elliptic_values(
        k
    )

    ratio1 = (
        Kp / K
    )

    ratio2 = (
        K / Kp
    )

    identity_value = (
        np.log(q)
        *
        np.log(qc)
    )

    # --------------------------------------------------------
    # Slider value
    # --------------------------------------------------------

    k_value.value = (
        f'<div class="nome-value">{k:.2f}</div>'
    )

    # --------------------------------------------------------
    # Current values
    # --------------------------------------------------------

    kp_math.value = (
        r'\('
        r'k^{\prime}='
        +
        f'{kp:.6f}'
        +
        r'\)'
    )

    K_math.value = (
        r'\('
        r'K='
        +
        f'{K:.6f}'
        +
        r'\)'
    )

    Kp_math.value = (
        r'\('
        r'K^{\prime}='
        +
        f'{Kp:.6f}'
        +
        r'\)'
    )

    q_math.value = (
        r'\('
        r'q='
        +
        f'{q:.8f}'
        +
        r'\)'
    )

    qc_math.value = (
        r'\('
        r'q_c='
        +
        f'{qc:.8f}'
        +
        r'\)'
    )

    ratio1_math.value = (
        r'\('
        r'\dfrac{K^{\prime}}{K}='
        +
        f'{ratio1:.6f}'
        +
        r'\)'
    )

    ratio2_math.value = (
        r'\('
        r'\dfrac{K}{K^{\prime}}='
        +
        f'{ratio2:.6f}'
        +
        r'\)'
    )

    identity_math.value = (
        r'\('
        r'\ln(q)\ln(q_c)='
        +
        f'{identity_value:.12f}'
        +
        r'\)'
    )

    # --------------------------------------------------------
    # Update markers
    # --------------------------------------------------------

    current_nome_line.set_xdata(
        [k, k]
    )

    current_ratio_line.set_xdata(
        [k, k]
    )

    point_q.set_data(
        [k],
        [q]
    )

    point_qc.set_data(
        [k],
        [qc]
    )

    point_ratio1.set_data(
        [k],
        [ratio1]
    )

    point_ratio2.set_data(
        [k],
        [ratio2]
    )

    # --------------------------------------------------------
    # Series approximation
    # --------------------------------------------------------

    if k <= threshold:

        approximation = q_series(
            k
        )

        error = abs(
            q - approximation
        )

        series_exact_math.value = (
            r'\('
            r'q_{\mathrm{exact}}='
            +
            f'{q:.10f}'
            +
            r'\)'
        )

        series_approx_math.value = (
            r'\('
            r'q_{\mathrm{series}}='
            +
            f'{approximation:.10f}'
            +
            r'\)'
        )

    else:

        approximation = qc_series(
            k
        )

        error = abs(
            qc - approximation
        )

        series_exact_math.value = (
            r'\('
            r'q_{c,\mathrm{exact}}='
            +
            f'{qc:.10f}'
            +
            r'\)'
        )

        series_approx_math.value = (
            r'\('
            r'q_{c,\mathrm{series}}='
            +
            f'{approximation:.10f}'
            +
            r'\)'
        )

    series_error_math.value = (
        r'\('
        r'\mathrm{absolute\ error}='
        +
        f'{error:.3e}'
        +
        r'\)'
    )

    # --------------------------------------------------------
    # Redraw only
    # --------------------------------------------------------

    fig_nome.canvas.draw_idle()
    fig_ratio.canvas.draw_idle()

# ============================================================
# CONNECT CONTROL
# ============================================================

k_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        definitions_row,
        top_row,
        figures_row,
        bottom_row
    ],
    layout=Layout(
        width='1220px',
        gap='10px',
        overflow='visible'
    )
)

display(
    main_layout
)